# PCNet-M: resmî COCOA örneği

**Durum:** Resmî COCOA ağırlığı Mac'te tamamen indirildi ve SHA-256 kaydedildi.
Bu notebook'un CUDA çalıştırması henüz doğrulanmadı; kayıtlı tahmin çıktısı içermez.
Linux/Python 3.12 yardımcı paket çözümü ve wheel bağlantıları kontrol edildi.
Başarı ancak 6. hücre gerçek ağırlıklarla tahmin üretip `run.json` içinde `success` yazdığında doğrulanır.

1. Colab → **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. Sürüm seçeneği varsa **2026.07** seçin (resmî listede Python 3.12.13 / PyTorch 2.11.0).
3. Aşağıdaki **1–8 numaralı kod hücrelerini sırayla** çalıştırın. Hata olursa durun ve **8. hücreyi** çalıştırarak tanı ZIP'ini indirin.

Ücretsiz GPU kullanın; ücretli kaynak oluşturulmaz. Eğitim ve tam veri kümesi indirme yoktur.
PCNet-M'ye verilen görünür maskeler resmî örnek anotasyonlarından gelir; bu bir RGB'den otomatik nesne bulma sistemi değildir.
COCOA modeli `use_rgb=False` kullanır: ağın girdisi hedef maske ve örten nesne maskesidir.
GT amodal maskeler yalnızca sonuç karşılaştırmasında kullanılır.

[Resmî depo](https://github.com/XiaohangZhan/deocclusion) ·
[Ağırlık klasörü](https://drive.google.com/drive/folders/1O89ItVWucCoL_VxIbLM1XLxr9JFfyj_Y) ·
[Colab sürüm listesi](https://research.google.com/colaboratory/runtime-version-faq.html)

EfficientSAM-Ti'ye geçiş yapılmaz; onun standart ağırlıkları amodal değildir ve ayrı amodal ince ayar gerektirir.

## 1 — Ortamı kontrol et

Bu hücre gerçek Python/PyTorch/CUDA sürümlerini kaydeder. Python 3.12 ve Linux x86_64 gereklidir.
PyTorch 2.11.0 dışında durur; uygun 2026.07 çalışma zamanını seçin. Eski PyTorch veya CUDA kurmaz.
Bu sürüm seçimi **çalıştırılarak doğrulanmış bir uyumluluk iddiası değildir**; sabitlenmiş başlangıç adayıdır.

In [ ]:
from pathlib import Path
import os, sys, json, platform, subprocess, hashlib, shutil, uuid
from datetime import datetime, timezone

ROOT = Path('/content/pcnet-m-start')
ROOT.mkdir(parents=True, exist_ok=True)
LOGS = ROOT / 'logs'
LOGS.mkdir(exist_ok=True)
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S') + '_' + uuid.uuid4().hex[:6]
OUT = ROOT / 'outputs' / SESSION

def command(argv, name, env=None, cwd=None):
    print('Çalıştırılıyor:', ' '.join(map(str, argv)), flush=True)
    log = LOGS / (SESSION + '_' + name + '.log')
    with log.open('w') as stream:
        proc = subprocess.Popen(list(map(str, argv)), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, env=env, cwd=cwd)
        for line in proc.stdout:
            print(line, end='')
            stream.write(line)
        result = proc.wait()
    if result:
        raise RuntimeError(f'{name} başarısız (kod {result}). Log: {log}. 8. hücre ile tanı indirin.')

try:
    import torch
    environment = {'python': sys.version, 'platform': platform.platform(),
                   'torch': torch.__version__, 'cuda_build': torch.version.cuda,
                   'cuda_available': torch.cuda.is_available(),
                   'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
except Exception as exc:
    environment = {'python': sys.version, 'platform': platform.platform(), 'torch_error': repr(exc)}
(LOGS / (SESSION + '_environment.json')).write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))
assert sys.platform == 'linux' and platform.machine() == 'x86_64', 'Linux x86_64 Colab ortamı gerekiyor.'
assert sys.version_info[:2] == (3, 12), 'Bu kilit Python 3.12 için. Colab 2026.07 sürümünü seçin.'
assert environment.get('torch', '').split('+')[0] == '2.11.0', 'Colab 2026.07 / PyTorch 2.11.0 seçin; burada PyTorch indirmiyoruz.'
assert environment.get('cuda_available'), 'GPU yok. Çalışma zamanı türünden ücretsiz T4 GPU seçin.'

## 2 — Sabit commit ve yardımcı dosyalar

Resmî depo `third_party/deocclusion` altında tutulur. Kopyası Mac'te yaklaşık 39 MB yer kapladı.
Notebook kendine yeterlidir; Mac'ten script veya veri yüklemeniz gerekmez. Aşağıdaki gömülü dosyalar
projedeki `scripts/` dosyalarından üretilmiştir. Kaynak kodu indirmesi tam veri kümesini içermez.

In [ ]:
BUNDLE = {'scripts/prepare_runtime.py': '"""Build a small inference-only view of pinned upstream code; never edit third_party.\n\nOriginal architecture, forward pass and inference algorithms are preserved.\nNumPy aliases and tensor device placement are adapted in inference.py. Exact official\nread_COCOA and expand_bbox functions are extracted to avoid training/demo imports.\n"""\nimport argparse\nimport ast\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport subprocess\n\nCOMMIT = \'ac543f9a54f4cb8baf0774bdb5f8638eadf8b57c\'\n\n\ndef extract(source, name):\n    tree = ast.parse(source)\n    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == name)\n    return ast.get_source_segment(source, node) + \'\\n\'\n\n\ndef prepare(repo, destination):\n    repo, destination = Path(repo).resolve(), Path(destination).resolve()\n    if destination == repo or repo in destination.parents or destination in repo.parents:\n        raise ValueError(\'Çalışma kopyası resmî depodan ayrı olmalı.\')\n    head = subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip()\n    dirty = subprocess.check_output([\'git\', \'-C\', str(repo), \'status\', \'--porcelain\'], text=True)\n    if head != COMMIT or dirty:\n        raise RuntimeError(f\'Resmî depo commit/temizlik kontrolü başarısız: {head}\\n{dirty}\')\n    destination.mkdir(parents=True, exist_ok=True)\n    hashes = {}\n\n    def write(relative, source):\n        target = destination / relative\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_text(source)\n\n    def read(relative):\n        raw = (repo / relative).read_bytes()\n        hashes[relative] = hashlib.sha256(raw).hexdigest()\n        return raw.decode()\n\n    for file in [\'models/backbone/unet/unet_model.py\', \'models/backbone/unet/unet_parts.py\',\n                 \'models/backbone/others.py\', \'utils/data_utils.py\', \'LICENSE\']:\n        write(file, read(file))\n    write(\'models/__init__.py\', \'# Generated inference-only package.\\n\')\n    write(\'models/backbone/__init__.py\', \'from .unet import unet2\\nfrom .others import FixModule\\n\')\n    write(\'models/backbone/unet/__init__.py\', \'from .unet_model import unet2\\n\')\n    write(\'utils/__init__.py\', \'from .data_utils import *\\n\')\n    source = read(\'inference.py\')\n    counts = {}\n    for old, new in [(\'int\', \'int\'), (\'bool\', \'bool\'), (\'float\', \'float\')]:\n        source, counts[old] = re.subn(r\'\\bnp\\.\' + old + r\'\\b\', new, source)\n    device_calls = source.count(\'.cuda()\')\n    if device_calls != 8:\n        raise RuntimeError(\'Unexpected upstream device calls; review adaptation.\')\n    source = source.replace(\'.cuda()\', \'.to(next(model.model.parameters()).device)\')\n    write(\'inference.py\', \'# Generated compatibility copy; see runtime_manifest.json.\\n\' + source)\n    helper = \'# Functions extracted unchanged from the pinned Apache-2.0 source.\\n\'\n    helper += \'import numpy as np\\nimport pycocotools.mask as maskUtils\\nimport utils\\n\\n\'\n    helper += extract(read(\'datasets/reader.py\'), \'read_COCOA\') + \'\\n\'\n    helper += extract(read(\'demos/demo_utils.py\'), \'expand_bbox\')\n    write(\'official_helpers.py\', helper)\n    report = {\'commit\': head, \'source_sha256\': hashes, \'numpy_alias_replacements\': counts,\n              \'scope\': \'COCOA PCNet-M inference only; not a training environment\',\n              \'architecture_changed\': False, \'cuda_calls_preserved\': False,\n              \'device_placement_replacements\': device_calls,\n              \'device_policy\': \'Input tensors follow the model device; no algorithm changes\'}\n    write(\'runtime_manifest.json\', json.dumps(report, indent=2))\n    return report\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--repo\', default=\'third_party/deocclusion\')\n    parser.add_argument(\'--output\', default=\'.runtime/deocclusion\')\n    args = parser.parse_args()\n    print(json.dumps(prepare(args.repo, args.output), indent=2))\n', 'scripts/download_weights.py': '"""Download exactly the official COCOA PCNet-M checkpoint; reject HTML/partial files."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport subprocess\n\n\ndef verify(path, manifest):\n    path = Path(path)\n    if path.stat().st_size != manifest[\'bytes\']:\n        raise ValueError(f"Boyut yanlış: {path.stat().st_size}; beklenen {manifest[\'bytes\']}")\n    with path.open(\'rb\') as stream:\n        digest = hashlib.file_digest(stream, \'sha256\').hexdigest()\n    if digest != manifest[\'sha256\']:\n        raise ValueError(f"SHA-256 uyuşmuyor: {digest}. Dosyayı model olarak yüklemeyin.")\n    return digest\n\n\ndef download(manifest_path, destination):\n    manifest = json.loads(Path(manifest_path).read_text())\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        print(\'Mevcut ağırlık doğrulandı:\', verify(destination, manifest))\n        return\n    print(f"Resmî COCOA PCNet-M: {manifest[\'bytes\']:,} bayt; yalnızca maske tahmini.", flush=True)\n    if manifest[\'bytes\'] > 500_000_000:\n        raise RuntimeError(\'500 MB sınırı: indirme başlatılmadı.\')\n    partial = destination.with_name(destination.name + \'.part\')\n    subprocess.run([\n        \'curl\', \'--location\', \'--fail\', \'--show-error\', \'--connect-timeout\', \'30\',\n        \'--max-time\', \'600\', \'--max-filesize\', str(manifest[\'bytes\']),\n        \'--output\', str(partial), manifest[\'download_url\'],\n    ], check=True)\n    digest = verify(partial, manifest)\n    partial.replace(destination)\n    print(\'İndirme ve SHA-256 doğrulandı:\', digest)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--manifest\', default=\'weights/manifest.json\')\n    parser.add_argument(\'--output\', default=\'weights/COCOA_pcnet_m.pth.tar\')\n    args = parser.parse_args()\n    download(args.manifest, args.output)\n', 'scripts/run_pcnet_m.py': '"""Run the official COCOA PCNet-M inference algorithms on one bundled example.\n\nSupports CPU and NVIDIA CUDA. The unchanged official UNet and FixModule are loaded\ndirectly, avoiding training optimizers and unrelated PCNet-C/matting imports.\n"""\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport traceback\nimport time\nfrom types import SimpleNamespace\n\nfrom download_weights import verify\n\n\ndef file_hash(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef run(args, report):\n    started = time.perf_counter()\n    # No writes to global Matplotlib caches; compatible with the local sandbox.\n    os.environ.setdefault(\'MPLCONFIGDIR\', str(args.output / \'mpl-cache\'))\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    import numpy as np\n    from PIL import Image\n    import pycocotools.mask as mask_utils\n    import torch\n    import yaml\n\n    report.update(torch=torch.__version__, cuda_build=torch.version.cuda,\n                  cuda_available=torch.cuda.is_available())\n    device = args.device\n    if device == \'auto\':\n        device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    if device == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA requested but unavailable. Use --device cpu.\')\n    torch.set_num_threads(args.threads)\n    report[\'device\'] = device\n    report[\'threads\'] = args.threads\n    report[\'gpu\'] = torch.cuda.get_device_name(0) if device == \'cuda\' else None\n    report[\'packages\'] = {name: importlib.metadata.version(name) for name in [\n        \'numpy\', \'Pillow\', \'matplotlib\', \'pycocotools\', \'PyYAML\', \'scikit-image\']}\n    # Record portable versions, without local wheel paths or user-directory names.\n    installed = json.loads(subprocess.check_output(\n        [sys.executable, \'-m\', \'pip\', \'list\', \'--format=json\'], text=True))\n    report[\'pip_freeze\'] = sorted(f"{p[\'name\']}=={p[\'version\']}" for p in installed)\n    torch.manual_seed(0)\n    np.random.seed(0)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n\n    sys.path.insert(0, str(args.runtime.resolve()))\n    import inference as infer\n    from models.backbone import unet2, FixModule\n    from official_helpers import read_COCOA, expand_bbox\n\n    config_path = args.repo / \'experiments/COCOA/pcnet_m/config.yaml\'\n    config = yaml.safe_load(config_path.read_text())\n    params = config[\'model\']\n    if (params[\'backbone_arch\'] != \'unet2\' or params[\'use_rgb\'] is not False\n            or params[\'backbone_param\'] != {\'in_channels\': 2, \'n_classes\': 2}):\n        raise RuntimeError(\'Beklenen resmî COCOA PCNet-M yapılandırması değil.\')\n    manifest = json.loads(args.manifest.read_text())\n    report[\'checkpoint_sha256\'] = verify(args.weights, manifest)\n    # Explicit safe tensor loading; never fall back to a random model on failure.\n    checkpoint = torch.load(args.weights, map_location=\'cpu\', weights_only=True)\n    if not isinstance(checkpoint, dict) or \'state_dict\' not in checkpoint:\n        raise RuntimeError(\'Checkpoint state_dict içermiyor.\')\n    net = FixModule(unet2(**params[\'backbone_param\']))\n    state = checkpoint[\'state_dict\']\n    if set(state) != set(net.state_dict()):\n        missing = sorted(set(net.state_dict()) - set(state))\n        extra = sorted(set(state) - set(net.state_dict()))\n        raise RuntimeError(f\'Ağırlık anahtarları uyuşmuyor. Eksik: {missing}; fazla: {extra}\')\n    net.load_state_dict(state, strict=True)\n    report[\'checkpoint_load\'] = \'strict=True; all keys matched\'\n    report[\'checkpoint_step\'] = int(checkpoint[\'step\']) if \'step\' in checkpoint else None\n    del state, checkpoint\n    for tensor in net.state_dict().values():\n        if not torch.isfinite(tensor).all().item():\n            raise RuntimeError(\'Ağırlıklarda sonlu olmayan değer bulundu.\')\n    net = net.to(device).eval()\n    model = SimpleNamespace(model=net)\n\n    image_path = args.repo / f\'demos/demo_data/COCOA/{args.example}.jpg\'\n    annot_path = image_path.with_suffix(\'.json\')\n    image = np.asarray(Image.open(image_path).convert(\'RGB\'))\n    height, width = image.shape[:2]\n    ann = json.loads(annot_path.read_text())\n    modal, category, boxes, gt = [], [], [], []\n    for region in ann[\'regions\']:\n        visible, box, cat = read_COCOA(region, height, width)\n        if not visible.any():\n            raise RuntimeError(\'Tamamen görünmez nesne bulundu; bu başlangıç örneği desteklemiyor.\')\n        modal.append(visible)\n        boxes.append(box)\n        category.append(cat)\n        gt.append(mask_utils.decode(mask_utils.merge(\n            mask_utils.frPyObjects([region[\'segmentation\']], height, width))))\n    modal, category, boxes, gt = map(np.asarray, (modal, category, boxes, gt))\n    bboxes = expand_bbox(boxes, enlarge_ratio=3., single_ratio=1.5)\n    calls = []\n\n    def check_logits(module, inputs, output):\n        if not torch.isfinite(output).all().item():\n            raise RuntimeError(\'Tahmin logits değerleri NaN/Inf içeriyor.\')\n        calls.append(list(output.shape))\n\n    hook = net.register_forward_hook(check_logits)\n    with torch.inference_mode():\n        order = infer.infer_order(\n            model, image, modal, category, bboxes, use_rgb=False, th=0.1,\n            dilate_kernel=0, input_size=256, min_input_size=16,\n            interp=\'nearest\', debug_info=False)\n        predicted_patches = infer.infer_amodal(\n            model, image, modal, category, bboxes, order, use_rgb=False, th=0.5,\n            dilate_kernel=0, input_size=256, min_input_size=16,\n            interp=\'linear\', debug_info=False)\n        predicted = infer.patch_to_fullimage(\n            predicted_patches, bboxes, height, width, interp=\'linear\')\n    hook.remove()\n    if not calls or predicted.shape != modal.shape or not np.isin(predicted, [0, 1]).all():\n        raise RuntimeError(\'Gerçek model çağrısı / tahmin boyutu / ikili maske kontrolü başarısız.\')\n\n    # Per-instance sample diagnostics, never a full-dataset benchmark claim.\n    metrics = []\n    for index, (visible, prediction, target) in enumerate(zip(modal, predicted, gt)):\n        visible, prediction, target = (m.astype(bool) for m in (visible, prediction, target))\n        def iou(a, b):\n            union = int((a | b).sum())\n            return float((a & b).sum() / union) if union else 1.0\n        hidden = target & ~visible\n        added = prediction & ~visible\n        metrics.append({\'instance\': index, \'visible_pixels\': int(visible.sum()),\n                        \'predicted_pixels\': int(prediction.sum()), \'hidden_gt_pixels\': int(hidden.sum()),\n                        \'added_pixels\': int(added.sum()),\n                        \'lost_visible_pixels\': int((visible & ~prediction).sum()),\n                        \'amodal_iou\': iou(prediction, target), \'visible_baseline_iou\': iou(visible, target),\n                        \'hidden_iou\': iou(added, hidden) if hidden.any() else None})\n    (args.output / \'metrics.json\').write_text(json.dumps(metrics, indent=2))\n\n    np.savez_compressed(args.output / \'masks.npz\', modal=modal,\n                        amodal_pred=predicted, amodal_gt=gt, order_matrix=order,\n                        bboxes=bboxes, category=category)\n    Image.fromarray(image).save(args.output / \'input.png\')\n    for label, masks in [(\'modal\', modal), (\'amodal_pred\', predicted), (\'amodal_gt\', gt)]:\n        folder = args.output / label\n        folder.mkdir(exist_ok=True)\n        for index, mask in enumerate(masks):\n            Image.fromarray((mask * 255).astype(np.uint8)).save(folder / f\'{index:02d}.png\')\n\n    colors = np.random.default_rng(0).uniform(0.2, 1., (len(modal), 3))\n    fig, axes = plt.subplots(1, 3, figsize=(15, 6), constrained_layout=True)\n    for ax, masks, title in zip(axes, [modal, predicted, gt],\n                               [\'Girdi: gorunur maskeler\', \'PCNet-M: amodal tahmin\', \'Referans: amodal GT\']):\n        ax.imshow(image)\n        for index, mask in enumerate(masks):\n            rgba = np.zeros((height, width, 4), dtype=np.float32)\n            rgba[..., :3] = colors[index]\n            rgba[..., 3] = mask * 0.45\n            ax.imshow(rgba)\n        ax.set_title(title)\n        ax.axis(\'off\')\n    fig.savefig(args.output / \'comparison.png\', dpi=150)\n    plt.close(fig)\n    # Select an occluded object for a legible binary-mask comparison. GT is only\n    # used for this display selection, never passed to infer_order/infer_amodal.\n    selected = int(((gt == 1) & (modal == 0)).sum(axis=(1, 2)).argmax())\n    fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)\n    for ax, mask, title in zip(axes, [modal[selected], predicted[selected], gt[selected]],\n                              [\'Gorunur maske\', \'PCNet-M tahmini\', \'Referans GT\']):\n        ax.imshow(mask, cmap=\'gray\', vmin=0, vmax=1)\n        ax.set_title(f\'Nesne {selected}: {title}\')\n        ax.axis(\'off\')\n    fig.savefig(args.output / \'object_comparison.png\', dpi=150)\n    plt.close(fig)\n    report.update(status=\'success\', example=args.example, instances=len(modal),\n                  image_shape=list(image.shape), model_forward_calls=len(calls),\n                  parameters=sum(p.numel() for p in net.parameters()),\n                  input_image_sha256=file_hash(image_path), annotation_sha256=file_hash(annot_path),\n                  runner_sha256=file_hash(__file__),\n                  config_sha256=file_hash(config_path),\n                  runtime=json.loads((args.runtime / \'runtime_manifest.json\').read_text()),\n                  thresholds={\'order\': 0.1, \'amodal\': 0.5},\n                  input_size=256, seed=0, ground_truth_used_for_inference=False,\n                  prediction_postprocessing=\'Official patch_to_fullimage; no extra union with modal masks\',\n                  elapsed_seconds=round(time.perf_counter() - started, 3),\n                  mean_amodal_iou=float(np.mean([m[\'amodal_iou\'] for m in metrics])),\n                  mean_visible_baseline_iou=float(np.mean([m[\'visible_baseline_iou\'] for m in metrics])),\n                  limitation=\'Bundled demonstration sample; not a held-out dataset benchmark.\')\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--repo\', type=Path, default=Path(\'third_party/deocclusion\'))\n    parser.add_argument(\'--runtime\', type=Path, default=Path(\'.runtime/deocclusion\'))\n    parser.add_argument(\'--manifest\', type=Path, default=Path(\'weights/manifest.json\'))\n    parser.add_argument(\'--weights\', type=Path, default=Path(\'weights/COCOA_pcnet_m.pth.tar\'))\n    parser.add_argument(\'--example\', type=int, choices=range(1, 6), default=4)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--device\', choices=[\'auto\', \'cpu\', \'cuda\'], default=\'auto\')\n    parser.add_argument(\'--threads\', type=int, default=4)\n    args = parser.parse_args()\n    if args.threads < 1:\n        parser.error(\'--threads must be positive\')\n    # Prevent stale masks from looking like the result of a failed new run.\n    if args.output.exists() and any(args.output.iterdir()):\n        raise RuntimeError(\'Çıktı klasörü dolu. Yeni bir çıktı klasörü seçin.\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    report = {\'status\': \'started\', \'python\': sys.version, \'platform\': platform.platform(),\n              \'timestamp_utc\': datetime.now(timezone.utc).isoformat()}\n    try:\n        run(args, report)\n        print(\'GERÇEK TAHMİN TAMAMLANDI:\', args.output)\n    except Exception:\n        report.update(status=\'failed\', traceback=traceback.format_exc())\n        raise\n    finally:\n        (args.output / \'run.json\').write_text(json.dumps(report, indent=2, ensure_ascii=False))\n\n\nif __name__ == \'__main__\':\n    main()\n', 'weights/manifest.json': '{\n  "checked_at": "2026-09-07",\n  "source_folder": "https://drive.google.com/drive/folders/1O89ItVWucCoL_VxIbLM1XLxr9JFfyj_Y",\n  "filename": "COCOA_pcnet_m.pth.tar",\n  "file_id": "12T_PKk5vB59bbLRil7dJrlAP6oJZipny",\n  "download_url": "https://drive.google.com/uc?export=download&id=12T_PKk5vB59bbLRil7dJrlAP6oJZipny",\n  "bytes": 26870559,\n  "sha256": "3740e8f709f824068dd36be46bd44ffd18105c4f606c099789c93f3ae4b860f7",\n  "download_verified": true,\n  "verification": "curl exit 0; HTTP 200; Content-Length matched; SHA-256 calculated; legacy torch pickle header inspected without unpickling",\n  "torch_load_verified": true,\n  "inference_verified": true,\n  "hash_origin": "Computed locally from the official download; not a publisher-provided checksum.",\n  "inference_verified_at": "2026-09-30",\n  "inference_device": "cpu",\n  "inference_evidence": "docs/results/verification.json"\n}\n', 'third_party/deocclusion.lock.json': '{\n  "repository": "https://github.com/XiaohangZhan/deocclusion.git",\n  "commit": "ac543f9a54f4cb8baf0774bdb5f8638eadf8b57c",\n  "checked_at": "2026-09-07",\n  "license": "Apache-2.0",\n  "configuration": "experiments/COCOA/pcnet_m/config.yaml",\n  "demo": "demos/demo_cocoa.ipynb"\n}\n', 'requirements-colab.lock.txt': '# CPython 3.12 / Linux x86_64: exact wheel URLs and hashes.\n# Metadata resolution verified; CUDA execution has NOT been verified.\n# contourpy==1.3.2\ncontourpy @ https://files.pythonhosted.org/packages/a8/32/b8a1c8965e4f72482ff2d1ac2cd670ce0b542f203c8e1d34e7c3e6925da7/contourpy-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:f26b383144cf2d2c29f01a1e8170f50dacf0eac02d64139dcd709a8ac4eb3cfe\n# cycler==0.12.1\ncycler @ https://files.pythonhosted.org/packages/e7/05/c19819d5e3d95294a6f5947fb9b9629efb316b96de511b418c53d245aae6/cycler-0.12.1-py3-none-any.whl --hash=sha256:85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30\n# fonttools==4.64.0\nfonttools @ https://files.pythonhosted.org/packages/dc/8f/e5f2906ea833d362916c64a2f6b1922a07b19442744fd3cd89563f429bff/fonttools-4.64.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl --hash=sha256:66a83f93579fb3493e458c4449d1d566a7b2a1c7b19915cd0fa3c9b8b5a8540b\n# ImageIO==2.37.4\nImageIO @ https://files.pythonhosted.org/packages/3e/2d/ca050652104bab2cf55e569db2a178b1b61cb041fef28307f2db383f6d9f/imageio-2.37.4-py3-none-any.whl --hash=sha256:1ab2e22c8debf700f24c3ac43e8f95f3b3a8110c83b93411e97b4b0b2cd1c7e6\n# kiwisolver==1.5.1\nkiwisolver @ https://files.pythonhosted.org/packages/fc/f4/dadfec469313c7f428efa7e84b4aba9732f813c13ea7131a24b7b008ef57/kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl --hash=sha256:34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38\n# lazy-loader==0.5\nlazy-loader @ https://files.pythonhosted.org/packages/8a/a1/8d812e53a5da1687abb10445275d41a8b13adb781bbf7196ddbcf8d88505/lazy_loader-0.5-py3-none-any.whl --hash=sha256:ab0ea149e9c554d4ffeeb21105ac60bed7f3b4fd69b1d2360a4add51b170b005\n# matplotlib==3.10.3\nmatplotlib @ https://files.pythonhosted.org/packages/c4/91/ba0ae1ff4b3f30972ad01cd4a8029e70a0ec3b8ea5be04764b128b66f763/matplotlib-3.10.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:ed70453fd99733293ace1aec568255bc51c6361cb0da94fa5ebf0649fdb2150a\n# networkx==3.6.1\nnetworkx @ https://files.pythonhosted.org/packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a4/networkx-3.6.1-py3-none-any.whl --hash=sha256:d47fbf302e7d9cbbb9e2555a0d267983d2aa476bac30e90dfbe5669bd57f3762\n# numpy==2.0.2\nnumpy @ https://files.pythonhosted.org/packages/39/68/e9f1126d757653496dbc096cb429014347a36b228f5a991dae2c6b6cfd40/numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:0123ffdaa88fa4ab64835dcbde75dcdf89c453c922f18dced6e27c90d1d0ec5a\n# opencv-python-headless==4.11.0.86\nopencv-python-headless @ https://files.pythonhosted.org/packages/dd/5c/c139a7876099916879609372bfa513b7f1257f7f1a908b0bdc1c2328241b/opencv_python_headless-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:0e0a27c19dd1f40ddff94976cfe43066fbbe9dfbb2ec1907d66c19caef42a57b\n# packaging==26.3\npackaging @ https://files.pythonhosted.org/packages/63/34/ba1c580383c9eada3711951fef0795c80b829a078d72188184bcab9dd527/packaging-26.3-py3-none-any.whl --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n# pillow==11.3.0\npillow @ https://files.pythonhosted.org/packages/8d/33/e9200d2bd7ba00dc3ddb78df1198a6e80d7669cce6c2bdbeb2530a74ec58/pillow-11.3.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6\n# pycocotools==2.0.10\npycocotools @ https://files.pythonhosted.org/packages/49/80/912b4c60f94e747dd2c3adbda5d4a4edc1d735fbfa0d91ab2eb231decb5d/pycocotools-2.0.10-cp312-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:4539d8b29230de042f574012edd0b5227528da083c4f12bbd6488567aabd3920\n# pyparsing==3.3.2\npyparsing @ https://files.pythonhosted.org/packages/10/bd/c038d7cc38edc1aa5bf91ab8068b63d4308c66c4c8bb3cbba7dfbc049f9c/pyparsing-3.3.2-py3-none-any.whl --hash=sha256:850ba148bd908d7e2411587e247a1e4f0327839c40e2e5e6d05a007ecc69911d\n# python-dateutil==2.9.0.post0\npython-dateutil @ https://files.pythonhosted.org/packages/ec/57/56b9bcc3c9c6a792fcbaf139543cee77261f3651ca9da0c93f5c1221264b/python_dateutil-2.9.0.post0-py2.py3-none-any.whl --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n# PyYAML==6.0.2\nPyYAML @ https://files.pythonhosted.org/packages/b9/2b/614b4752f2e127db5cc206abc23a8c19678e92b23c3db30fc86ab731d3bd/PyYAML-6.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:80bab7bfc629882493af4aa31a4cfa43a4c57c83813253626916b8c7ada83476\n# scikit-image==0.25.2\nscikit-image @ https://files.pythonhosted.org/packages/6b/b5/b75527c0f9532dd8a93e8e7cd8e62e547b9f207d4c11e24f0006e8646b36/scikit_image-0.25.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --hash=sha256:a17e17eb8562660cc0d31bb55643a4da996a81944b82c54805c91b3fe66f4824\n# scipy==1.16.3\nscipy @ https://files.pythonhosted.org/packages/79/2e/415119c9ab3e62249e18c2b082c07aff907a273741b3f8160414b0e9193c/scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl --hash=sha256:72d1717fd3b5e6ec747327ce9bda32d5463f472c9dce9f54499e81fbd50245a1\n# six==1.17.0\nsix @ https://files.pythonhosted.org/packages/b7/ce/149a00dd41f10bc29e5921b496af8b574d8413afcd5e30dfa0ed46c2cc5e/six-1.17.0-py2.py3-none-any.whl --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274\n# tifffile==2026.4.11\ntifffile @ https://files.pythonhosted.org/packages/3f/9f/74f110b4271ded519c7add4341cbabc824de26817ff1c345b3109df9e99c/tifffile-2026.4.11-py3-none-any.whl --hash=sha256:9b94ffeddb39e97601af646345e8808f885773de01b299e480ed6d3a41509ec9\n', 'reports/colab-wheels.json': '[\n  {\n    "name": "numpy",\n    "version": "2.0.2",\n    "url": "https://files.pythonhosted.org/packages/39/68/e9f1126d757653496dbc096cb429014347a36b228f5a991dae2c6b6cfd40/numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "0123ffdaa88fa4ab64835dcbde75dcdf89c453c922f18dced6e27c90d1d0ec5a"\n  },\n  {\n    "name": "pillow",\n    "version": "11.3.0",\n    "url": "https://files.pythonhosted.org/packages/8d/33/e9200d2bd7ba00dc3ddb78df1198a6e80d7669cce6c2bdbeb2530a74ec58/pillow-11.3.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl",\n    "sha256": "67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6"\n  },\n  {\n    "name": "matplotlib",\n    "version": "3.10.3",\n    "url": "https://files.pythonhosted.org/packages/c4/91/ba0ae1ff4b3f30972ad01cd4a8029e70a0ec3b8ea5be04764b128b66f763/matplotlib-3.10.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "ed70453fd99733293ace1aec568255bc51c6361cb0da94fa5ebf0649fdb2150a"\n  },\n  {\n    "name": "PyYAML",\n    "version": "6.0.2",\n    "url": "https://files.pythonhosted.org/packages/b9/2b/614b4752f2e127db5cc206abc23a8c19678e92b23c3db30fc86ab731d3bd/PyYAML-6.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "80bab7bfc629882493af4aa31a4cfa43a4c57c83813253626916b8c7ada83476"\n  },\n  {\n    "name": "pycocotools",\n    "version": "2.0.10",\n    "url": "https://files.pythonhosted.org/packages/49/80/912b4c60f94e747dd2c3adbda5d4a4edc1d735fbfa0d91ab2eb231decb5d/pycocotools-2.0.10-cp312-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "4539d8b29230de042f574012edd0b5227528da083c4f12bbd6488567aabd3920"\n  },\n  {\n    "name": "opencv-python-headless",\n    "version": "4.11.0.86",\n    "url": "https://files.pythonhosted.org/packages/dd/5c/c139a7876099916879609372bfa513b7f1257f7f1a908b0bdc1c2328241b/opencv_python_headless-4.11.0.86-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "0e0a27c19dd1f40ddff94976cfe43066fbbe9dfbb2ec1907d66c19caef42a57b"\n  },\n  {\n    "name": "scikit-image",\n    "version": "0.25.2",\n    "url": "https://files.pythonhosted.org/packages/6b/b5/b75527c0f9532dd8a93e8e7cd8e62e547b9f207d4c11e24f0006e8646b36/scikit_image-0.25.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "a17e17eb8562660cc0d31bb55643a4da996a81944b82c54805c91b3fe66f4824"\n  },\n  {\n    "name": "contourpy",\n    "version": "1.3.2",\n    "url": "https://files.pythonhosted.org/packages/a8/32/b8a1c8965e4f72482ff2d1ac2cd670ce0b542f203c8e1d34e7c3e6925da7/contourpy-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "f26b383144cf2d2c29f01a1e8170f50dacf0eac02d64139dcd709a8ac4eb3cfe"\n  },\n  {\n    "name": "cycler",\n    "version": "0.12.1",\n    "url": "https://files.pythonhosted.org/packages/e7/05/c19819d5e3d95294a6f5947fb9b9629efb316b96de511b418c53d245aae6/cycler-0.12.1-py3-none-any.whl",\n    "sha256": "85cef7cff222d8644161529808465972e51340599459b8ac3ccbac5a854e0d30"\n  },\n  {\n    "name": "fonttools",\n    "version": "4.64.0",\n    "url": "https://files.pythonhosted.org/packages/dc/8f/e5f2906ea833d362916c64a2f6b1922a07b19442744fd3cd89563f429bff/fonttools-4.64.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl",\n    "sha256": "66a83f93579fb3493e458c4449d1d566a7b2a1c7b19915cd0fa3c9b8b5a8540b"\n  },\n  {\n    "name": "ImageIO",\n    "version": "2.37.4",\n    "url": "https://files.pythonhosted.org/packages/3e/2d/ca050652104bab2cf55e569db2a178b1b61cb041fef28307f2db383f6d9f/imageio-2.37.4-py3-none-any.whl",\n    "sha256": "1ab2e22c8debf700f24c3ac43e8f95f3b3a8110c83b93411e97b4b0b2cd1c7e6"\n  },\n  {\n    "name": "kiwisolver",\n    "version": "1.5.1",\n    "url": "https://files.pythonhosted.org/packages/fc/f4/dadfec469313c7f428efa7e84b4aba9732f813c13ea7131a24b7b008ef57/kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl",\n    "sha256": "34633ecf50d16187ab8e5528b7a2530f2feb4e23f300db4672538b51cfc5cd38"\n  },\n  {\n    "name": "lazy-loader",\n    "version": "0.5",\n    "url": "https://files.pythonhosted.org/packages/8a/a1/8d812e53a5da1687abb10445275d41a8b13adb781bbf7196ddbcf8d88505/lazy_loader-0.5-py3-none-any.whl",\n    "sha256": "ab0ea149e9c554d4ffeeb21105ac60bed7f3b4fd69b1d2360a4add51b170b005"\n  },\n  {\n    "name": "networkx",\n    "version": "3.6.1",\n    "url": "https://files.pythonhosted.org/packages/9e/c9/b2622292ea83fbb4ec318f5b9ab867d0a28ab43c5717bb85b0a5f6b3b0a4/networkx-3.6.1-py3-none-any.whl",\n    "sha256": "d47fbf302e7d9cbbb9e2555a0d267983d2aa476bac30e90dfbe5669bd57f3762"\n  },\n  {\n    "name": "packaging",\n    "version": "26.3",\n    "url": "https://files.pythonhosted.org/packages/63/34/ba1c580383c9eada3711951fef0795c80b829a078d72188184bcab9dd527/packaging-26.3-py3-none-any.whl",\n    "sha256": "d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c"\n  },\n  {\n    "name": "pyparsing",\n    "version": "3.3.2",\n    "url": "https://files.pythonhosted.org/packages/10/bd/c038d7cc38edc1aa5bf91ab8068b63d4308c66c4c8bb3cbba7dfbc049f9c/pyparsing-3.3.2-py3-none-any.whl",\n    "sha256": "850ba148bd908d7e2411587e247a1e4f0327839c40e2e5e6d05a007ecc69911d"\n  },\n  {\n    "name": "python-dateutil",\n    "version": "2.9.0.post0",\n    "url": "https://files.pythonhosted.org/packages/ec/57/56b9bcc3c9c6a792fcbaf139543cee77261f3651ca9da0c93f5c1221264b/python_dateutil-2.9.0.post0-py2.py3-none-any.whl",\n    "sha256": "a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427"\n  },\n  {\n    "name": "scipy",\n    "version": "1.16.3",\n    "url": "https://files.pythonhosted.org/packages/79/2e/415119c9ab3e62249e18c2b082c07aff907a273741b3f8160414b0e9193c/scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl",\n    "sha256": "72d1717fd3b5e6ec747327ce9bda32d5463f472c9dce9f54499e81fbd50245a1"\n  },\n  {\n    "name": "six",\n    "version": "1.17.0",\n    "url": "https://files.pythonhosted.org/packages/b7/ce/149a00dd41f10bc29e5921b496af8b574d8413afcd5e30dfa0ed46c2cc5e/six-1.17.0-py2.py3-none-any.whl",\n    "sha256": "4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274"\n  },\n  {\n    "name": "tifffile",\n    "version": "2026.4.11",\n    "url": "https://files.pythonhosted.org/packages/3f/9f/74f110b4271ded519c7add4341cbabc824de26817ff1c345b3109df9e99c/tifffile-2026.4.11-py3-none-any.whl",\n    "sha256": "9b94ffeddb39e97601af646345e8808f885773de01b299e480ed6d3a41509ec9"\n  }\n]\n'}
for relative, content in BUNDLE.items():
    path = ROOT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
lock = json.loads((ROOT / 'third_party/deocclusion.lock.json').read_text())
REPO = ROOT / 'third_party/deocclusion'
if not (REPO / '.git').exists():
    REPO.mkdir(parents=True, exist_ok=True)
    command(['git', 'init', REPO], 'git_init')
    command(['git', '-C', REPO, 'remote', 'add', 'origin', lock['repository']], 'git_remote')
    command(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', lock['commit']], 'git_fetch')
    command(['git', '-C', REPO, 'checkout', '--detach', lock['commit']], 'git_checkout')
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == lock['commit'], 'Commit uyuşmuyor; yeni Colab çalışma zamanı açın.'
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip()
print('Sabit resmî commit:', actual)

## 3 — Hafif yardımcı paketleri ayrı klasöre kur

Colab'ın PyTorch/CUDA kurulumu korunur. Yardımcı paketler yalnızca bu oturumdaki `support/` klasörüne
kurulur ve sonraki Python alt süreçleri bu klasörü kullanır. Notebook çekirdeğine paket yüklenmez;
yeniden başlatma gerekmez. Her paketin sürümü ve wheel SHA-256 değeri kilitlidir.

Önce indirme boyutları gösterilir. 500 MB üzerindeki veya boyutu belirlenemeyen wheel indirilmez.
Bu adım paket kurulumunu sınar; GPU tahmini 6. hücrededir.

In [ ]:
import urllib.request
SUPPORT = ROOT / 'support'
req = ROOT / 'requirements-colab.lock.txt'
req_hash = hashlib.sha256(req.read_bytes()).hexdigest()
marker = SUPPORT / '.complete'
if not marker.exists():
    if SUPPORT.exists() and any(SUPPORT.iterdir()):
        raise RuntimeError('Önceki kurulum yarım kalmış. Yeni Colab çalışma zamanı açıp tekrar çalıştırın.')
    items = json.loads((ROOT / 'reports/colab-wheels.json').read_text())
    total = 0
    for item in items:
        name = item['name']
        assert not name.lower().startswith(('torch', 'nvidia', 'triton')), name
        url = item['url']
        with urllib.request.urlopen(urllib.request.Request(url, method='HEAD'), timeout=30) as response:
            size = int(response.headers.get('Content-Length', '0'))
        print(f'{name}: {size / 1e6:.2f} MB — veri okuma / maske işleme / çizim bağımlılığı')
        assert 0 < size <= 500_000_000, f'{name}: boyut belirsiz veya 500 MB sınırını aşıyor; indirme durduruldu.'
        total += size
    print(f'Toplam yardımcı paket indirmesi: {total / 1e6:.1f} MB')
    command([sys.executable, '-m', 'pip', 'install', '--target', SUPPORT,
             '--only-binary=:all:', '--require-hashes', '--no-deps',
             '--report', LOGS / (SESSION + '_install_report.json'), '-r', req], 'dependency_install')
    marker.write_text(req_hash)
assert marker.read_text() == req_hash, 'Paket kilidi değişmiş; yeni Colab çalışma zamanı açın.'
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(SUPPORT)
ENV['MPLCONFIGDIR'] = str(ROOT / 'mpl-cache')
check = "import sys,torch,numpy,cv2,yaml,skimage,pycocotools; from skimage.morphology import convex_hull; from PIL import Image; import matplotlib; print(sys.version); print('torch',torch.__version__,'numpy',numpy.__version__,'cv2',cv2.__version__,'CUDA',torch.cuda.is_available()); assert torch.cuda.is_available(); x=torch.ones(1,device='cuda'); print('CUDA tensor check:',x.cpu().tolist())"
command([sys.executable, '-c', check], 'imports_and_cuda_tensor', ENV)

## 4 — Resmî ağırlığı indir ve doğrula

Yalnızca `COCOA_pcnet_m.pth.tar`: **26.870.559 bayt ≈ 26,9 MB**.
Bu dosyanın tamamı Mac'te 7 Eylül 2026 tarihinde indirildi.
Boyut ve SHA-256 eşleşmeden tahmine geçilmez. `.pth.tar` adına rağmen dosya bir PyTorch checkpoint'idir;
`tar` komutuyla açmayın. Drive erişimi başarısızsa HTML çıktısı ağırlık kabul edilmez.

SHA-256: `3740e8f709f824068dd36be46bd44ffd18105c4f606c099789c93f3ae4b860f7`.
Bu özet ilk resmî indirmeden yerelde hesaplandı; yayıncının sağladığı bir imza değildir.

In [ ]:
WEIGHTS = ROOT / 'weights/COCOA_pcnet_m.pth.tar'
command([sys.executable, ROOT / 'scripts/download_weights.py',
         '--manifest', ROOT / 'weights/manifest.json', '--output', WEIGHTS], 'weights_download')

## 5 — Uyumluluk kopyasını hazırla

Resmî UNet katmanları ve ileri geçiş kodu aynen alınır. Tahmin kopyasındaki eski `np.int`/`np.bool`
ifadeleri Python türleriyle değiştirilir. Girdi tensörleri modelin cihazına taşınır;
bu notebook CUDA cihazını kullanır, yerel demo ise CPU'yu da destekler.
`read_COCOA` ve `expand_bbox` resmî koddan aynen alınır.
YAML `safe_load` ile; ağırlık `weights_only=True`, `strict=True` ile yüklenir.
PCNet-C, matting, eğitim, optimizer ve eski `cvbase` bağımlılıkları bu tahmin akışına alınmaz.
Bu düzenlemeler tam resmî eğitim ortamına uyumluluk sağlamaz.

In [ ]:
RUNTIME = ROOT / '.runtime/deocclusion'
command([sys.executable, ROOT / 'scripts/prepare_runtime.py',
         '--repo', REPO, '--output', RUNTIME], 'prepare_runtime')

## 6 — Gerçek ağırlıklarla örnek tahmin

Varsayılan **örnek 4**, resmî COCOA demosunun seçimiyle aynıdır. İsterseniz sonraki çalıştırmada
`EXAMPLE` değerini 1–5 arasında değiştirin. Her çalıştırma ayrı klasör üretir.
Örtülme sırası eşiği 0,1; amodal eşiği 0,5; maske girdi boyutu 256×256'dır.
Ağ çağrıları ve sonlu çıktılar denetlenir. Model yüklenemezse işlem durur; yedek rastgele tahmin yoktur.

In [ ]:
EXAMPLE = 4
OUT = ROOT / 'outputs' / (SESSION + '_' + uuid.uuid4().hex[:6])
command([sys.executable, ROOT / 'scripts/run_pcnet_m.py',
         '--repo', REPO, '--runtime', RUNTIME, '--manifest', ROOT / 'weights/manifest.json',
         '--weights', WEIGHTS, '--example', EXAMPLE, '--device', 'cuda', '--output', OUT], 'inference', ENV)
run_report = json.loads((OUT / 'run.json').read_text())
assert run_report['status'] == 'success'
print('Gerçek tahmin doğrulandı. Ağ çağrısı:', run_report['model_forward_calls'])

## 7 — Maskeleri görselleştir

İlk görsel: görünür girdi maskeleri, PCNet-M amodal tahmini ve referans amodal maskeler.
İkinci görsel: örtülen bir nesnenin ikili maskeleri. GT yalnızca görsel seçim/karşılaştırma için kullanılır.
Tahmin referansla aynı olmak zorunda değildir. Tek örnek araştırma başarısını veya veri kümesi metriğini göstermez.

In [ ]:
from IPython.display import display, Image as DisplayImage
assert (OUT / 'run.json').exists(), 'Önce 6. hücreyi çalıştırın.'
assert json.loads((OUT / 'run.json').read_text())['status'] == 'success', 'Tahmin başarısız; 8. hücreden tanı indirin.'
display(DisplayImage(filename=str(OUT / 'comparison.png')))
display(DisplayImage(filename=str(OUT / 'object_comparison.png')))

## 8 — Sonuçları veya hata kayıtlarını indir

Hata aldıysanız da bu hücreyi çalıştırabilirsiniz (1. hücrenin başlamış olması gerekir).
Başarı varsa ZIP: `comparison.png`, `object_comparison.png`, `input.png`, nesne başına PNG maskeler,
`masks.npz`, `run.json`, kullanılan scriptler ve kurulum kayıtları içerir.
PNG maskeler 0/255; NPZ maskeler 0/1 değerindedir. Sıra matrisi: `-1` satır nesnesinin sütun nesnesinin
arkasında olduğunu gösterir. Hata varsa ZIP adı `pcnet_m_diagnostics_...` olur.
Ağırlıklar ve bütün depo ZIP'e eklenmez.

In [ ]:
import zipfile
from google.colab import files
success = (OUT / 'run.json').exists() and json.loads((OUT / 'run.json').read_text()).get('status') == 'success'
kind = 'results' if success else 'diagnostics'
archive = ROOT / f'pcnet_m_{kind}_{SESSION}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [LOGS, ROOT / 'scripts', OUT]:
        if folder.exists():
            for path in sorted(folder.rglob('*')):
                if path.is_file() and '__pycache__' not in path.parts and 'mpl-cache' not in path.parts:
                    z.write(path, path.relative_to(ROOT))
    for relative in ['weights/manifest.json', 'third_party/deocclusion.lock.json',
                     'requirements-colab.lock.txt', 'reports/colab-wheels.json',
                     '.runtime/deocclusion/runtime_manifest.json']:
        path = ROOT / relative
        if path.exists():
            z.write(path, relative)
print('İndiriliyor:', archive.name)
files.download(str(archive))